# Agentic Search Tutorial

Learn how to **build a searchable knowledge base** and **query it with natural language** using multi-round agentic reasoning.

## How It Works

Agentic search uses a multi-round LLM reasoning loop (Opie backbone) to find the most relevant chunks from your indexed documents. Unlike simple keyword or embedding search, the agent autonomously decides:
- Which internal tools to call (BM25, question matching, metadata filters)
- How many rounds of reasoning to perform
- When it has found enough relevant chunks to stop

## Available Model

| Model | Type | Description | Pricing | Speed |
|-------|------|-------------|---------|-------|
| **macchiato_v1** | Agentic Search | Multi-round LLM reasoning over indexed KBs | $0.12/search | ~2-5s |

## Workflow Overview

1. **Prepare data** - Structure your documents as chunks with metadata
2. **Create an index** - Upload data to build a searchable index (BM25 + embeddings)
3. **Search** - Query the index using the SDK `SearchClient`
4. **Clean up** - Delete the index when done

## 1. Installation

In [15]:
# Install from local source in editable mode
%pip install -e ../ python-dotenv --upgrade 

Obtaining file:///Users/oussama/Documents/Cmprsr/codebase/Compresr-SDK-Private/python
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for compresr (pyproject.toml) ... done
  Created wheel for compresr: filename=compresr-2.0.4-0.editable-py3-none-any.whl size=5715 sha256=c132fd0e56cc0b3c03b152c13086803861aebb12016db47a61bc808a7d31dfae
  Stored in directory: /private/var/folders/g4/rzc_8mgd15n5f0l73k0mm3600000gn/T/pip-ephem-wheel-cache-gmk9xfbq/wheels/b8/66/33/38ffb0bd34b6589c6f5f9d07abdc3fcffb34736427d2cdac72
Successfully built compresr
  Attempting uninstall: compresr
    Found existing installation: compresr 2.0.4
    Uninstalling compresr-2.0.4:
      Successfully uninstalled compresr-2.0.4
Note: you may need to restart the kernel to use updated packages.


## 2. Setup

In [16]:
import os
import json
from dotenv import load_dotenv
from compresr import SearchClient, MODELS

# Load environment variables
env_path = "../../.env"
load_dotenv(env_path)

api_key = os.getenv("COMPRESR_API_KEY")

# Default: https://api.compresr.ai (production)
# Uncomment for local development:
os.environ["COMPRESR_BASE_URL"] = "http://localhost:8000"

# Initialize SDK client for searching AND index management
client = SearchClient(api_key=api_key)

print("Client initialized!")
print(f"Base URL: {client._base_url}")
print(f"Search Model: {MODELS.MACCHIATO}")

Client initialized!
Base URL: http://localhost:8000
Search Model: macchiato_v1


In [17]:
# Sample dataset - a small knowledge base about AI topics
upload_data = {
    "person_name": "AI Tutor",
    "person_description": "An educational knowledge base about artificial intelligence.",
    "chunks": [
        {
            "_id": "chunk_001",
            "sourceId": "src_001",
            "order": 0,
            "text": "Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed. The three main types are supervised learning (labeled data), unsupervised learning (finding patterns), and reinforcement learning (learning from rewards)."
        },
        {
            "_id": "chunk_002",
            "sourceId": "src_001",
            "order": 1,
            "text": "Deep learning uses artificial neural networks with multiple layers to model complex patterns. Convolutional neural networks (CNNs) excel at image recognition, while recurrent neural networks (RNNs) and transformers handle sequential data like text and speech."
        },
        {
            "_id": "chunk_003",
            "sourceId": "src_002",
            "order": 0,
            "text": "Natural language processing (NLP) enables computers to understand and generate human language. Modern NLP relies on transformer architectures like BERT for understanding and GPT for generation. Key tasks include sentiment analysis, named entity recognition, and machine translation."
        },
        {
            "_id": "chunk_004",
            "sourceId": "src_002",
            "order": 1,
            "text": "Large language models (LLMs) are trained on massive text corpora and can perform a wide range of tasks through prompting. Techniques like few-shot learning, chain-of-thought reasoning, and retrieval-augmented generation (RAG) improve their effectiveness."
        },
        {
            "_id": "chunk_005",
            "sourceId": "src_003",
            "order": 0,
            "text": "Computer vision allows machines to interpret visual information from images and video. Applications include autonomous driving, medical image analysis, facial recognition, and quality control in manufacturing."
        },
    ],
    "candidate_questions": [
        {"_id": "q_001", "sourceId": "src_001", "chunkId": "chunk_001", "question": "What is machine learning?"},
        {"_id": "q_002", "sourceId": "src_001", "chunkId": "chunk_001", "question": "What are the types of machine learning?"},
        {"_id": "q_003", "sourceId": "src_001", "chunkId": "chunk_002", "question": "How does deep learning work?"},
        {"_id": "q_004", "sourceId": "src_001", "chunkId": "chunk_002", "question": "What are CNNs and RNNs?"},
        {"_id": "q_005", "sourceId": "src_002", "chunkId": "chunk_003", "question": "What is natural language processing?"},
        {"_id": "q_006", "sourceId": "src_002", "chunkId": "chunk_004", "question": "What are large language models?"},
        {"_id": "q_007", "sourceId": "src_003", "chunkId": "chunk_005", "question": "What is computer vision?"},
    ],
    "source_docs": {
        "src_001": {"_id": "src_001", "title": "ML and Deep Learning", "shortSummary": "Introduction to machine learning and deep learning concepts."},
        "src_002": {"_id": "src_002", "title": "NLP and LLMs", "shortSummary": "Natural language processing and large language models."},
        "src_003": {"_id": "src_003", "title": "Computer Vision", "shortSummary": "Visual AI and its applications."},
    },
    "chunk_summaries": {
        "chunk_001": "Overview of machine learning types: supervised, unsupervised, reinforcement.",
        "chunk_002": "Deep learning architectures: CNNs for images, RNNs/transformers for sequences.",
        "chunk_003": "NLP fundamentals using transformer models like BERT and GPT.",
        "chunk_004": "LLM capabilities and techniques: few-shot, CoT, RAG.",
        "chunk_005": "Computer vision applications in autonomous driving, medical imaging, etc.",
    },
}

print("Dataset prepared:")
print(f"   Chunks:    {len(upload_data['chunks'])}")
print(f"   Questions: {len(upload_data['candidate_questions'])}")
print(f"   Sources:   {len(upload_data['source_docs'])}")
print(f"   Summaries: {len(upload_data['chunk_summaries'])}")
print(f"   Payload:   {len(json.dumps(upload_data).encode()) / 1024:.1f} KB")

Dataset prepared:
   Chunks:    5
   Questions: 7
   Sources:   3
   Summaries: 5
   Payload:   3.3 KB


### Loading from CSV Files

For larger datasets, you typically load from CSV files. The expected file structure:

```
my-dataset/
  sourceChunksText.csv      # _id, sourceId, order, text
  sourceChunksQuestions.csv  # _id, sourceId, chunkId, question
  sources.csv               # _id, title, shortSummary
  person.csv                # name, shortDescription
  sourceChunksSummary.csv   # chunkId, summary (optional)
```

See `agentic_search.ipynb` for a full example using the `load_upload_payload()` helper with CSV data.

## 4. Create an Index

Upload your data to create a searchable index. This builds BM25 indexes, question embeddings,
and metadata structures that the agentic search model uses.

The server returns a `task_id` immediately (HTTP 202) and builds the index in the background.
Use `client.wait_for_index()` to poll until the build completes.

In [18]:
INDEX_NAME = "tutorial-ai-kb"

# Create the index using the SDK
task = client.create_index(index_name=INDEX_NAME, data=upload_data)

print("Index build started!")
print(f"   Task ID: {task['task_id']}")
print(f"   Status:  {task['status']}")

Index build started!
   Task ID: b66e50e839df47ef
   Status:  building


In [19]:
# Wait for the index to be built (polls automatically)
status = client.wait_for_index(task_id=task["task_id"])
info = status.get("result", {})

print("Index built!")
print(f"   Index name:  {info.get('index_name', INDEX_NAME)}")
print(f"   Chunks:      {info.get('num_chunks', '?')}")
print(f"   Questions:   {info.get('num_questions', '?')}")
print(f"   Sources:     {info.get('num_sources', '?')}")
print(f"   Embeddings:  {info.get('has_embeddings', '?')}")

Index built!
   Index name:  tutorial-ai-kb
   Chunks:      5
   Questions:   7
   Sources:     3
   Embeddings:  True


## 5. Basic Search

Now that the index is built, use the SDK `SearchClient` to query it with natural language.
The agentic model will reason over the index and return the most relevant chunks.

In [20]:
response = client.search(
    query="What is machine learning?",
    index_name=INDEX_NAME,
    compression_model_name=MODELS.MACCHIATO,
)

print(f"Chunk IDs returned: {response.data.chunks_count}")
if response.data.cost:
    print(f"Cost: ${response.data.cost['request_cost_usd']:.2f} (${response.data.cost['price_per_search']:.2f}/search)")
print(f"Duration: {response.data.duration_ms}ms")
print()

# Show retrieved chunks
for cid, chunk_text in zip(response.data.chunk_ids, response.data.chunks):
    text_preview = chunk_text[:200]
    print(f"--- {cid} ---")
    print(text_preview + "...")
    print()

Chunk IDs returned: 5
Cost: $0.12 ($0.12/search)
Duration: 1665ms

--- chunk_001 ---
Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed. The three main types are supervised learning (labeled data), unsupe...

--- chunk_002 ---
Deep learning uses artificial neural networks with multiple layers to model complex patterns. Convolutional neural networks (CNNs) excel at image recognition, while recurrent neural networks (RNNs) an...

--- chunk_003 ---
Natural language processing (NLP) enables computers to understand and generate human language. Modern NLP relies on transformer architectures like BERT for understanding and GPT for generation. Key ta...

--- chunk_004 ---
Large language models (LLMs) are trained on massive text corpora and can perform a wide range of tasks through prompting. Techniques like few-shot learning, chain-of-thought reasoning, and retrieval-a...

--- chunk_005 ---
Computer vision allows 

## 6. Multiple Searches

Run several queries against the same index to see how the agentic search
handles different topics and returns different relevant chunks.

In [21]:
questions = [
    "How does deep learning work?",
    "What are large language models and how are they used?",
    "What applications does computer vision have?",
]

for q in questions:
    r = client.search(query=q, index_name=INDEX_NAME)
    n_chunks = r.data.chunks_count
    cost = f", ${r.data.cost['request_cost_usd']:.2f}" if r.data.cost else ""
    print(f"Q: {q}")
    print(f"   {n_chunks} chunks{cost}")
    print()

Q: How does deep learning work?
   3 chunks, $0.12

Q: What are large language models and how are they used?
   5 chunks, $0.12

Q: What applications does computer vision have?
   5 chunks, $0.12



## 7. Custom Search Parameters

Control the search with `max_time_s` \u2014 the maximum time the agentic loop can spend reasoning.

- **Lower values** (e.g., 2.0s) = faster, fewer reasoning rounds
- **Higher values** (e.g., 10.0s) = more thorough, more reasoning rounds
- **Default**: 4.5s

In [22]:
print("Testing Different max_time_s Values\n")

for max_time in [2.0, 4.5, 10.0]:
    response = client.search(
        query="What is natural language processing?",
        index_name=INDEX_NAME,
        max_time_s=max_time,
    )
    cost = f", ${response.data.cost['request_cost_usd']:.2f}" if response.data.cost else ""
    print(f"{'='*50}")
    print(f"max_time_s = {max_time}")
    print(f"{'='*50}")
    print(f"Chunks:    {response.data.chunks_count}")
    print(f"Duration:  {response.data.duration_ms}ms{cost}")
    print()

Testing Different max_time_s Values

max_time_s = 2.0
Chunks:    5
Duration:  1910ms, $0.12

max_time_s = 4.5
Chunks:    5
Duration:  2066ms, $0.12

max_time_s = 10.0
Chunks:    5
Duration:  1969ms, $0.12



## 8. Inspecting the Response Object

The `SearchResponse` object contains the full result with metrics. Here's a detailed look at everything available:

In [23]:
# Full response inspection
response = client.search(
    query="What is machine learning?",
    index_name=INDEX_NAME,
    compression_model_name="macchiato_v1",
)

print(f"Success:           {response.success}")
print(f"Chunk IDs returned:{response.data.chunks_count}")
print(f"Duration:          {response.data.duration_ms}ms")
print(f"Index name:        {response.data.index_name}")
if response.data.cost:
    print(f"Cost:              ${response.data.cost['request_cost_usd']:.2f}")
print("\nRaw data dict:")
print(json.dumps(response.model_dump(), indent=2, default=str))

Success:           True
Chunk IDs returned:3
Duration:          1908ms
Index name:        tutorial-ai-kb
Cost:              $0.12

Raw data dict:
{
  "success": true,
  "message": null,
  "data": {
    "chunks": [
      "Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed. The three main types are supervised learning (labeled data), unsupervised learning (finding patterns), and reinforcement learning (learning from rewards).",
      "Deep learning uses artificial neural networks with multiple layers to model complex patterns. Convolutional neural networks (CNNs) excel at image recognition, while recurrent neural networks (RNNs) and transformers handle sequential data like text and speech.",
      "Natural language processing (NLP) enables computers to understand and generate human language. Modern NLP relies on transformer architectures like BERT for understanding and GPT for generation. Key tasks include s

## 9. Async Search

In [24]:
async def search_async_demo():
    response = await client.search_async(
        query="What are the types of machine learning?",
        index_name=INDEX_NAME,
        compression_model_name=MODELS.MACCHIATO,
    )
    print("Async search successful!")
    print(f"   Chunks:   {response.data.chunks_count}")
    print(f"   Duration: {response.data.duration_ms}ms")

await search_async_demo()

Async search successful!
   Chunks:   1
   Duration: 1899ms


## 10. Error Handling

In [25]:
from compresr.exceptions import (
    CompresrError,
    AuthenticationError,
    RateLimitError,
    ValidationError,
)

# Example: Successful search with error handling
try:
    response = client.search(
        query="Test query",
        index_name=INDEX_NAME,
    )
    print("Success!")
except AuthenticationError:
    print("Invalid API key")
except RateLimitError:
    print("Rate limit exceeded")
except ValidationError as e:
    print(f"Invalid input: {e}")
except CompresrError as e:
    print(f"API error: {e}")

Success!


### Common Error Scenarios

Test how the SDK handles various error conditions:

In [26]:
from compresr import SearchClient

# Error 1: Unsupported model name
print("1. Testing unsupported model name...")
try:
    client.search(query="Test", index_name=INDEX_NAME, compression_model_name="invalid_model_v1")
    print("Unexpected success\n")
except ValidationError as e:
    print(f"   Caught ValidationError: {e}\n")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {e}\n")

# Error 2: Empty query
print("2. Testing empty query...")
try:
    client.search(query="", index_name=INDEX_NAME)
    print("Unexpected success\n")
except ValidationError as e:
    print(f"   Caught ValidationError: {e}\n")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {e}\n")

# Error 3: Empty index_name
print("3. Testing empty index_name...")
try:
    client.search(query="Test", index_name="")
    print("Unexpected success\n")
except ValidationError as e:
    print(f"   Caught ValidationError: {e}\n")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {e}\n")

# Error 4: Invalid API key format
print("4. Testing invalid API key format...")
try:
    bad_client = SearchClient(api_key="sk-invalid-key")
    bad_client.search(query="Test", index_name=INDEX_NAME)
    print("Unexpected success\n")
except AuthenticationError as e:
    print(f"   Caught AuthenticationError: {e}\n")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {e}\n")

# Error 5: Wrong model for SearchClient (compression model)
print("5. Testing wrong model for client type...")
try:
    client.search(query="Test", index_name=INDEX_NAME, compression_model_name=MODELS.ESPRESSO)
    print("Unexpected success\n")
except ValidationError as e:
    print(f"   Caught ValidationError: {e}\n")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {e}\n")

print("All error scenarios tested!")

1. Testing unsupported model name...
   Caught ValidationError: Model 'invalid_model_v1' is not valid for SearchClient. Allowed models: macchiato_v1

2. Testing empty query...
   Caught ValidationError: 1 validation error for SearchRequest
query
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short

3. Testing empty index_name...
   Caught ValidationError: 1 validation error for SearchRequest
index_name
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short

4. Testing invalid API key format...
   Caught AuthenticationError: Invalid API key format. Keys must start with 'cmp_'

5. Testing wrong model for client type...
   Caught ValidationError: Model 'espresso_v1' is not valid for SearchClient. Allowed models: macchiato_v1

All err

## 11. Clean Up

Delete the index when you're done to free resources.

In [27]:
# Delete the index using the SDK
result = client.delete_index(index_name=INDEX_NAME)
print(result)

{'success': True, 'message': 'Index deleted'}


In [28]:
# Verify the API is healthy
print(client.health())

{'status': 'healthy', 'compression_server': True}


## Next Steps

- Docs: [docs.compresr.ai](https://docs.compresr.ai)
- Support: founders@compresr.ai
- Try agnostic compression: `agnostic_compression_tutorial.ipynb`
- Try question-specific compression: `question_specific_compression_tutorial.ipynb`
- See the raw Opie API tutorial: `agentic_search.ipynb`